# FGBuster and Furax Imports for Component Separation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/your-repo/your-notebook.ipynb)


In [1]:
# FGBUSTER IMPORTS
import healpy as hp
import pysm3

from fgbuster import (CMB, Dust, Synchrotron, 
                      basic_comp_sep, MixingMatrix,
                      get_observation, get_instrument)
from fgbuster.visualization import corner_norm

# FURAX IMPORTS
import jax
import jaxopt
import jax.numpy as jnp
from jax import ShapeDtypeStruct

from furax._base.blocks import BlockDiagonalOperator, BlockRowOperator
from furax._base.core import HomothetyOperator, IdentityOperator
from furax.landscapes import StokesPyTree, ValidStokesType, HealpixLandscape
from furax.tree import as_structure
from furax.operators.sed import CMBOperator, DustOperator, SynchrotronOperator , MixingMatrixOperator
import operator
from math import prod
import numpy as np
from functools import partial

import os
import pickle

## Mixed Sky Maps Creation Using PySM

In this section, we create simulated sky maps using the `PySM` library with specified parameters for each astrophysical component. Key elements:

- **NSIDE**: Sets the HEALPix resolution, determining the number of pixels in the sky map.
- **Reference Frequencies**:
  - **Dust** at 150 GHz
  - **Synchrotron** at 20 GHz
- **Instrument Configuration**: Using the `LiteBIRD` instrument model to simulate observed frequency maps.

This setup provides the mixed sky maps required for component separation, with the shape of the output maps indicated for verification.

`generate_maps` is used on Jean-Zay to cache the frequency maps on the front node since there is no internet access in the slurm nodes

In [2]:
instrument = get_instrument('LiteBIRD')

def generate_maps(nside):
    npixel = nside ** 2 * 12
    # Define cache file path
    cache_dir = 'freq_maps_cache'
    os.makedirs(cache_dir, exist_ok=True)
    cache_file = os.path.join(cache_dir, f'freq_maps_nside_{nside}.pkl')

    # Check if file exists, load if it does, otherwise create and save it
    if os.path.exists(cache_file):
        with open(cache_file, 'rb') as f:
            freq_maps = pickle.load(f)
        print(f"Loaded freq_maps for nside {nside} from cache.")
    else:
        # Generate freq_maps if not already cached
        freq_maps = get_observation(instrument, 'c1d0s0', nside=nside)
        
        # Save freq_maps to the cache
        with open(cache_file, 'wb') as f:
            pickle.dump(freq_maps, f)
        print(f"Generated and saved freq_maps for nside {nside}.")

    # Check the shape of freq_maps
    print("freq_maps shape:", freq_maps.shape)


nsides = [32 , 64 , 128 , 256 , 512]
for nside in nsides:
    generate_maps(nside)

Loaded freq_maps for nside 32 from cache.
freq_maps shape: (15, 3, 12288)
Loaded freq_maps for nside 64 from cache.
freq_maps shape: (15, 3, 49152)
Loaded freq_maps for nside 128 from cache.
freq_maps shape: (15, 3, 196608)
Loaded freq_maps for nside 256 from cache.
freq_maps shape: (15, 3, 786432)
Loaded freq_maps for nside 512 from cache.
freq_maps shape: (15, 3, 3145728)


In [3]:
nside = 32
npixel = nside ** 2 * 12
dust_nu0 = 150.0
synchrotron_nu0 = 20.0
stokes_type: ValidStokesType = 'IQU'
instrument = get_instrument('LiteBIRD')

# Define cache file path
cache_dir = 'freq_maps_cache'
cache_file = os.path.join(cache_dir, f'freq_maps_nside_{nside}.pkl')

# Check if file exists and load if it does; otherwise raise an error with guidance
if os.path.exists(cache_file):
    with open(cache_file, 'rb') as f:
        freq_maps = pickle.load(f)
    print(f"Loaded freq_maps for nside {nside} from cache.")
else:
    raise FileNotFoundError(
        f"Cache file for freq_maps with nside {nside} not found.\n"
        f"Please generate it first by calling `generate_maps({nside})`."
    )

# Check the shape of freq_maps
print("freq_maps shape:", freq_maps.shape)

Loaded freq_maps for nside 32 from cache.
freq_maps shape: (15, 3, 12288)


Furax expects a `StokesPyTree` object as input, so we convert the frequency maps to the correct format, so we transform the `freq_maps` into a `StokesPyTree` object to make it compatible with Furax.

__Note__: Although Furax includes its own functions to create sky maps from PySM, we use `fgbuster` here to ensure that both methods receive identical inputs for comparison.


In [4]:
d = StokesPyTree.from_stokes(I=freq_maps[:,0,:], Q=freq_maps[:,1,:], U=freq_maps[:,2,:])
structure = HealpixLandscape(nside, stokes_type).structure
d.structure

StokesIQUPyTree(i=ShapeDtypeStruct(shape=(15, 12288), dtype=float64), q=ShapeDtypeStruct(shape=(15, 12288), dtype=float64), u=ShapeDtypeStruct(shape=(15, 12288), dtype=float64))

In [5]:
components = [CMB(), Dust(dust_nu0), Synchrotron(synchrotron_nu0)]

## Defining the Likelihood Function for Component Separation

In this cell, we define the `negative_log_prob` function, which calculates the negative log-likelihood of observing the given data `d` based on the model parameters.

The likelihood function is based on a quadratic form that includes the mixing matrix `A`, inverse noise covariance `N^{-1}`, and observed data `d`. The key term in the likelihood is:

$$
\left(A^T N^{-1} d\right)^T \left(A^T N^{-1} A\right)^{-1} \left(A^T N^{-1} d\right)
$$

### Explanation of Each Term

1. **$A$**: The mixing matrix operator, which maps the component space to the observed frequency space.
2. **$N^{-1}$**: The inverse of the noise covariance matrix, represented by `invN` in the code.
3. **$d$**: The observed data, which is structured as a `StokesPyTree` in Furax.

### Implementation Details

- **Transposing and Applying `A`**: `A.T(d)` applies the transpose of `A` to `d`, equivalent to the term $A^T d$.
- **Computing the Likelihood**: The quadratic form is computed by applying $A^T N^{-1} d$, inverting $A^T N^{-1} A$, and performing matrix multiplications to evaluate the likelihood.
- **Negative Log-Likelihood**: The final output of `negative_log_prob` is the negative of this log-likelihood value, allowing us to use it as a loss function for optimization.


In [6]:
invN = HomothetyOperator(jnp.ones(1), _in_structure=d.structure)
DND = invN(d) @ d

in_structure = HealpixLandscape(nside, stokes_type).structure
best_params = {'temp_dust': 20., 'beta_dust': 1.54, 'beta_pl': -3.0}
nu = instrument['frequency'].values

@jax.jit
def negative_log_prob(params, d):
    cmb = CMBOperator(nu, in_structure=in_structure)
    dust = DustOperator(
        nu,
        frequency0=dust_nu0,
        temperature=params['temp_dust'],
        beta=params['beta_dust'],
        in_structure=in_structure,
    )
    synchrotron = SynchrotronOperator(nu,
                                      frequency0=synchrotron_nu0,
                                      beta_pl=params['beta_pl'],
                                      in_structure=in_structure)

    A =  MixingMatrixOperator(cmb=cmb, dust=dust, synchrotron=synchrotron)

    x = (A.T @ invN)(d)
    l = jax.tree.map(lambda a, b: a @ b, x, (A.T @ invN @ A).I(x))
    summed_log_prob = jax.tree.reduce(operator.add, l)

    return -summed_log_prob 

Evaluate the performance of the likelihood

In [7]:
print(f"Performance of the nll evaluation")
negative_log_prob(best_params, d).block_until_ready()
%timeit negative_log_prob(best_params, d).block_until_ready()
print(f"Performance of the nll grad evaluation")
jax.grad(negative_log_prob)(best_params, d)['beta_pl'].block_until_ready()
%timeit jax.grad(negative_log_prob)(best_params, d)['beta_pl'].block_until_ready()

Performance of the nll evaluation
3.06 ms ± 47.3 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)
Performance of the nll grad evaluation
9.56 ms ± 69.9 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


## Check for Correctness

In this cell, we perform a basic correctness check by comparing the gradients of the negative log-likelihood at two sets of parameters:

1. **Wrong Parameters**: A set of parameters obtained by adding random noise to `best_params`.
2. **Correct Parameters**: The original `best_params`.

By calculating and comparing the gradient magnitudes (using the `max` reduction), we can verify that the gradient at the correct parameters is smaller, indicating proximity to an optimal or near-optimal point.


In [8]:
wrong_params = jax.tree.map(lambda x: x + jax.random.normal(jax.random.PRNGKey(0)), best_params)
print(f"Wrong parameters grad {jax.tree.reduce(max , jax.grad(negative_log_prob)(wrong_params, d))}")
print(f"Correct parameters grad {jax.tree.reduce(max , jax.grad(negative_log_prob)(best_params, d))}")

Wrong parameters grad -1118221116.6310368
Correct parameters grad 154.78900920557885


## Off the shelf likelihoods



In [9]:
from furax.comp_sep import spectral_log_likelihood, spectral_cmb_variance, negative_log_likelihood

negative_log_likelihood = partial(negative_log_likelihood , dust_nu0=dust_nu0, synchrotron_nu0=synchrotron_nu0)

L = negative_log_likelihood(best_params,
                            nu=nu,
                            N=invN,
                            d=d)

assert jax.tree.all(
    jax.tree.map(lambda x, y: jnp.isclose(x, y, rtol=1e-5), L,
                 negative_log_prob(best_params, d)))

# Validating Against FGBuster: The `c1d0s0` Model

In this section, we validate our custom implementation of the likelihood model by comparing it to the `c1d0s0` model from `fgbuster`. By aligning our results with FGBuster’s well-established component separation model, we ensure that our setup and computations are consistent and reliable.


### Case 1 : Initial Validation: Using `best_params` as the Starting Point

We begin the validation process by setting `best_params` as the initial point for both our custom implementation and FGBuster’s `c1d0s0` model. This allows us to directly compare the outputs and confirm that the models produce similar results when initialized wit


In [10]:
components[1]._set_default_of_free_symbols(beta_d=1.54,temp=20.)
components[2]._set_default_of_free_symbols(beta_pl=-3.0)

result = basic_comp_sep(components, instrument, freq_maps)
print(result.params)
print(result.x)

['Dust.beta_d', 'Dust.temp', 'Synchrotron.beta_pl']
[ 1.54 20.   -3.  ]


In [11]:
best_params = {'temp_dust': 20., 'beta_dust': 1.54, 'beta_pl': -3.0}

scipy_solver = jaxopt.ScipyMinimize(fun=negative_log_likelihood, method='TNC', jit=True ,tol=1e-10)
result = scipy_solver.run(best_params, nu=nu, N=invN, d=d)
result.params

/home/wassim/micromamba/envs/jax/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.53999997, dtype=float64),
 'beta_pl': Array(-3., dtype=float64),
 'temp_dust': Array(19.99999992, dtype=float64)}

## Case 2 : Validation with Incorrect Parameter: Setting `beta_dust` to a Wrong Value

In [12]:
components[1]._set_default_of_free_symbols(beta_d=2.54,temp=20.)
components[2]._set_default_of_free_symbols(beta_pl=-3.0)

result = basic_comp_sep(components, instrument, freq_maps)
print(result.params)
print(result.x)

<lambdifygenerated-9>:2: RuntimeWarning: overflow encountered in power
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)
<lambdifygenerated-9>:2: RuntimeWarning: overflow encountered in power
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)
<lambdifygenerated-10>:2: RuntimeWarning: overflow encountered in power
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)*log(0.05*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)
<lambdifygenerated-9>:2: RuntimeWarning: overflow encountered in multiply
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)


SVD of A failed -> logL = -inf
SVD of A failed -> logL_dB not updated
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
['Dust.beta_d', 'Dust.temp', 'Synchrotron.beta_pl']
[ 1.53194524 19.97377854 -2.9430962 ]


In [13]:
params = {'temp_dust': 20., 'beta_dust': 2.54, 'beta_pl': -3.0}

scipy_solver = jaxopt.ScipyMinimize(fun=negative_log_likelihood, method='TNC' , jit=True ,tol=1e-10)
result = scipy_solver.run(params, nu=nu, N=invN, d=d)
result.params

/home/wassim/micromamba/envs/jax/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.54001987, dtype=float64),
 'beta_pl': Array(-3.00051503, dtype=float64),
 'temp_dust': Array(19.99937388, dtype=float64)}

## Case 3 : Setting `beta_dust` and `beta_pl` to Incorrect Values


In [14]:
components[1]._set_default_of_free_symbols(beta_d=2.54,temp=20.)
components[2]._set_default_of_free_symbols(beta_pl=-6.0)

result = basic_comp_sep(components, instrument, freq_maps)
print(result.params)
print(result.x)

<lambdifygenerated-9>:2: RuntimeWarning: overflow encountered in power
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)
<lambdifygenerated-9>:2: RuntimeWarning: overflow encountered in power
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)
<lambdifygenerated-10>:2: RuntimeWarning: overflow encountered in power
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)*log(0.05*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)
<lambdifygenerated-9>:2: RuntimeWarning: overflow encountered in multiply
  return 568.8620443215493*(0.05*nu)**beta_pl*numpy.expm1(0.01760867023799751*nu)**2*exp(-0.01760867023799751*nu)/(nu**2*numpy.expm1(0.3521734047599502)**2)


SVD of A failed -> logL = -inf
SVD of A failed -> logL_dB not updated
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
SVD of A failed -> logL = -inf
['Dust.beta_d', 'Dust.temp', 'Synchrotron.beta_pl']
[ 1.53034346 19.97418092 -5.99479857]


In [15]:
params = {'temp_dust': 20., 'beta_dust': 2.54, 'beta_pl': -6.0}

scipy_solver = jaxopt.ScipyMinimize(fun=negative_log_likelihood, method='TNC' , jit=True ,tol=1e-10)
result = scipy_solver.run(params, nu=nu, N=invN, d=d)
result.params

/home/wassim/micromamba/envs/jax/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.53990453, dtype=float64),
 'beta_pl': Array(-2.99892243, dtype=float64),
 'temp_dust': Array(20.00318078, dtype=float64)}

## Case 4 : Setting All Parameters to Incorrect Values


In [16]:
components[1]._set_default_of_free_symbols(beta_d=2.54,temp=25.)
components[2]._set_default_of_free_symbols(beta_pl=-6.0)

result = basic_comp_sep(components, instrument, freq_maps)
print(result.params)
print(result.x)

['Dust.beta_d', 'Dust.temp', 'Synchrotron.beta_pl']
[ 1.53999951 20.00001728 -2.99999135]


In [17]:
params = {'temp_dust': 25., 'beta_dust': 2.54, 'beta_pl': -6.0}

scipy_solver = jaxopt.ScipyMinimize(fun=negative_log_likelihood, method='TNC' , jit=True ,tol=1e-10)
result = scipy_solver.run(params, nu=nu, N=invN, d=d)
result.params

/home/wassim/micromamba/envs/jax/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.53999549, dtype=float64),
 'beta_pl': Array(-2.99992223, dtype=float64),
 'temp_dust': Array(20.00014915, dtype=float64)}

# Using Optax optimizers

In [18]:
from furax.comp_sep import optimize
import optax
import optax.tree_utils as otu

In [19]:
solver = optax.lbfgs()

final_params, final_state = optimize(params,
                                     negative_log_likelihood,
                                     solver,
                                     max_iter=100,
                                     tol=1e-4,
                                     nu=nu,
                                     N=invN,
                                     d=d)

print(
    f"Final parameters: {final_params}, number of evaluations: {otu.tree_get(final_state, 'count')}"
)
print(f"Initial Value: {negative_log_prob(final_params , d=d)}")


Final parameters: {'beta_dust': Array(1.54000014, dtype=float64), 'beta_pl': Array(-3.00000083, dtype=float64), 'temp_dust': Array(19.99999516, dtype=float64)}, number of evaluations: 38
Initial Value: -6114624736980.275


## Minimizing the CMB Variance



In [26]:
spectral_cmb_variance = partial(spectral_cmb_variance , dust_nu0=dust_nu0, synchrotron_nu0=synchrotron_nu0)


solver = optax.lbfgs()

cmb_var_final_params, cmb_var_final_state = optimize(params,
                                     spectral_cmb_variance,
                                     solver,
                                     max_iter=100,
                                     tol=1e-8,
                                     nu=nu,
                                     N=invN,
                                     d=d)

print(
    f"Final parameters: {cmb_var_final_params}, number of evaluations: {otu.tree_get(cmb_var_final_state, 'count')}"
)
print(f"Initial Value: {negative_log_prob(final_params , d=d)}")


Final parameters: {'beta_dust': Array(1.45637327, dtype=float64), 'beta_pl': Array(-3.03860917, dtype=float64), 'temp_dust': Array(24.98589264, dtype=float64)}, number of evaluations: 25
Initial Value: -6114624736980.275


In [21]:
scipy_solver = jaxopt.ScipyMinimize(fun=spectral_cmb_variance, method='TNC', jit=True ,tol=1e-10)
cmb_var_result = scipy_solver.run(params, nu=nu, N=invN, d=d)
cmb_var_result.params

/home/wassim/micromamba/envs/jax/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.46793765, dtype=float64),
 'beta_pl': Array(-3.03860699, dtype=float64),
 'temp_dust': Array(24.14901712, dtype=float64)}

The parameters : 

 - $\beta_d$ = 1.54
 - $T_d$ = 20.0
 - $\beta_pl$ = -3.0

Do minimize the likelihood .. but not the CMB Variance in case $d$ containts the intensity stokes parameter

How ever using only the polarization data, minimizing the CMB Variance and the Spectral Likelihood gives the same result

In [32]:
d_polarization = StokesPyTree.from_stokes(Q=freq_maps[:,1,:], U=freq_maps[:,2,:])
N_polarization = HomothetyOperator(jnp.ones(1), _in_structure=d_polarization.structure)

d_polarization.structure

StokesQUPyTree(q=ShapeDtypeStruct(shape=(15, 12288), dtype=float64), u=ShapeDtypeStruct(shape=(15, 12288), dtype=float64))

In [33]:
scipy_solver = jaxopt.ScipyMinimize(fun=spectral_cmb_variance, method='TNC', jit=True ,tol=1e-10 , maxiter=1000)
cmb_var_result = scipy_solver.run(params, nu=nu, N=N_polarization, d=d_polarization)
cmb_var_result.params

/home/wassim/micromamba/envs/jax/lib/python3.10/site-packages/jaxopt/_src/scipy_wrappers.py:343: OptimizeWarning: Unknown solver options: maxiter
  res = osp.optimize.minimize(scipy_fun, jnp_to_onp(init_params, self.dtype),


{'beta_dust': Array(1.46683482, dtype=float64),
 'beta_pl': Array(-3.05749423, dtype=float64),
 'temp_dust': Array(24.15524135, dtype=float64)}

In [35]:
solver = optax.lbfgs()
cmb_var_final_params, cmb_var_final_state = optimize(params,
                                     spectral_cmb_variance,
                                     solver,
                                     max_iter=100,
                                     tol=1e-8,
                                     nu=nu,
                                     N=N_polarization,
                                     d=d_polarization)

print(
    f"Final parameters: {final_params}, number of evaluations: {otu.tree_get(final_state, 'count')}"
)
print(f"Initial Value: {negative_log_prob(final_params , d=d)}")


Final parameters: {'beta_dust': Array(1.54000014, dtype=float64), 'beta_pl': Array(-3.00000083, dtype=float64), 'temp_dust': Array(19.99999516, dtype=float64)}, number of evaluations: 38
Initial Value: -6114624736980.275
